In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import torch
import os

import matplotlib.pyplot as plt
import hydra

import wandb

sys.path = ["..", *sys.path]

In [2]:
from matplotlib.colors import LogNorm
import deeptime as dt
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt

SELECTION = "symbol == C or symbol == N or symbol == S"


def distances(xyz):
    distance_matrix_ca = np.linalg.norm(xyz[:, None, :, :] - xyz[:, :, None, :], axis=-1)
    n_ca = distance_matrix_ca.shape[-1]
    m, n = np.triu_indices(n_ca, k=1)
    distances_ca = distance_matrix_ca[:, m, n]
    return distances_ca


def wrap(array):
    return (np.sin(array), np.cos(array))


def tica_features(trajectory, use_dihedrals=True, use_distances=True, selection=SELECTION):
    trajectory = trajectory.atom_slice(trajectory.top.select(selection))
    n_atoms = trajectory.xyz.shape[1]
    if use_dihedrals:
        _, phi = md.compute_phi(trajectory)
        _, psi = md.compute_psi(trajectory)
        _, omega = md.compute_omega(trajectory)
        dihedrals = np.concatenate([*wrap(phi), *wrap(psi), *wrap(omega)], axis=-1)
    if use_distances:
        ca_distances = distances(trajectory.xyz)
    if use_distances and use_dihedrals:
        return np.concatenate([ca_distances, dihedrals], axis=-1)
    elif use_distances:
        return ca_distances
    elif use_dihedrals:
        return ca_dihedrals
    else:
        return []


def plot_tic01(ax, tics, name, tics_lims, cmap="viridis"):
    _ = ax.hist2d(tics[:, 0], tics[:, 1], bins=100, norm=LogNorm(), cmap=cmap, rasterized=True)
    ax.set_xlabel("TIC0")  # , fontsize=45)
    ax.set_ylabel("TIC1")  # , fontsize=45)
    ax.set_ylim(tics_lims[:, 1].min(), tics_lims[:, 1].max())
    ax.set_xlim(tics_lims[:, 0].min(), tics_lims[:, 0].max())
    ax.set_xticks([])
    ax.set_yticks([])
    # plt.title(f"{name}", fontsize=45)
    return ax


def run_tica(trajectory, lagtime=500, dim=40):
    ca_features = tica_features(trajectory)
    tica = dt.decomposition.TICA(dim=dim, lagtime=lagtime)
    koopman_estimator = dt.covariance.KoopmanWeightingEstimator(lagtime=lagtime)
    reweighting_model = koopman_estimator.fit(ca_features).fetch_model()
    tica_model = tica.fit(ca_features, reweighting_model).fetch_model()
    return tica_model

In [3]:
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)

In [28]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["plot_me"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [59]:
import math

In [29]:
# df[[a for a in df.columns if "data/n" in a]]
DEST_FILE = (
    "/home/mila/b/bosejoey/scratch/fast-tbg/al2/test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al3/test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al4/test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al6/test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al2/jarzynski_test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al3/jarzynski_test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al4/jarzynski_test_samples.pt"
    "/home/mila/b/bosejoey/scratch/fast-tbg/al6/jarzynski_test_samples.pt"
)

In [60]:
def plot_samples(
    row, clip=0.002, overwrite=False, plot_energy=False, plot_rama=False, plot_3d=False
):
    model = "Unknown"
    if "matching" in row["model/_target_"]:
        if row["model/sigma"] == 0.01:
            model = "ecnf"
        else:
            model = "ecnf++"
    else:
        model = "SBG"
    if model != "SBG":
        print("Skipping non SBG")
        return
    data_dir = row["data/data_dir"]
    n_particles = row["data/n_particles"]
    print(n_particles)
    save_path = f"figs/density_v2/{n_particles}_{model}_density.png"
    rama_save_path = f"figs/rama_v2/{n_particles}_{model}_rama.png"
    tica_save_path = f"figs/tica_v2/{n_particles}_{model}_rama.png"
    if os.path.exists(save_path) and not overwrite:
        print(f"Skipping {save_path}")
        return
    if "bose" in data_dir or "ubuntu" in data_dir:
        data_dir = "../data/alanine"
        if n_particles == 22:
            data_dir = "../data/AD2/"
    try:
        sample_file = torch.load(f"{row['paths/output_dir']}/test_samples.pt", weights_only=True)
        if model == "SBG":
            jarz_sample_file = torch.load(
                f"{row['paths/output_dir']}/test_jarzynski_samples.pt", weights_only=True
            )
    except Exception as e:
        # try loading via joey scratch
        if n_particles == 63:
            residues = 6
        if n_particles == 42:
            residues = 4
        if n_particles == 33:
            residues = 3
        if n_particles < 30:
            residues = 2
        path = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_samples.pt"
        jpath = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_jarzynski_samples.pt"
        sample_file = torch.load(path, weights_only=True)
        jarz_sample_file = torch.load(jpath, weights_only=True)
        # print(e)
        # print(f"File not found {row['paths/output_dir']}/test_samples.pt. Skipping.")

    samples = sample_file["samples"]
    logp = sample_file["log_p"]
    jarzynski_samples = None
    if model == "SBG":
        jarzynski_samples = jarz_sample_file["samples"]
        jarzynski_logits = jarz_sample_file["logits"]
        clip = 0
    cfg = {
        "_target_": row["data/_target_"],
        "data_dir": data_dir,
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": int(row["data/n_particles"]),
        "n_dimensions": row["data/n_dimensions"],
        "dim": int(row["data/dim"]),
    }
    if n_particles > 22 and row["data/filename"] == row["data/filename"]:
        cfg["filename"] = row["data/filename"]
        cfg["pdb_filename"] = row["data/pdb_filename"]
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    min_energy = -50
    max_energy = 20
    ylim = (0, 0.1)
    if n_particles == 63:
        min_energy = -130
        max_energy = -40
        ylim = (0, 0.1)
    if n_particles == 53:
        min_energy = -150
        max_energy = -60
        ylim = (0, 0.1)
    if n_particles == 42:
        min_energy = -20
        max_energy = 80
        ylim = (0, 0.1)
    if n_particles == 33:
        min_energy = -200
        max_energy = -100
        ylim = (0, 0.1)
    # max_energy = 100
    if plot_energy:
        datamodule.plot_nice_samples(
            samples,
            log_p_samples=logp,
            samples_jarzynski=jarzynski_samples,
            min_energy=min_energy,
            max_energy=max_energy,
            clip_energy=max_energy,
            ylim=ylim,
        )
        print(f"saving plot to {save_path}")
        plt.savefig(save_path, dpi=300)
        plt.close()

    # ramaplot
    from matplotlib.colors import LogNorm

    # symmetry check
    from src.models.components.utils import (
        check_symmetry_change,
        compute_chirality_sign,
        find_chirality_centers,
        resample,
    )

    atom_dict = {"C": 0, "H": 1, "N": 2, "O": 3, "S": 4}
    atom_types = []
    for atom_name in datamodule.topology.atoms:
        atom_types.append(atom_name.name[0])
    atom_types = torch.from_numpy(np.array([atom_dict[atom_type] for atom_type in atom_types]))
    n_particles = len(atom_types)
    adj_list = torch.from_numpy(
        np.array(
            [(b.atom1.index, b.atom2.index) for b in datamodule.topology.bonds], dtype=np.int32
        )
    )
    if model == "SBG":
        samples = jarzynski_samples
        logits = jarzynski_logits
    else:
        logits = -energy_samples.flatten() - logp.flatten()
    energy_samples = datamodule.energy(samples)
    if clip > 0:
        clipped_logits_mask = logits > torch.quantile(logits, 1 - clip)
        logits = logits[~clipped_logits_mask]
        samples = samples[~clipped_logits_mask]
        energy_samples = energy_samples[~clipped_logits_mask]
    # model samples
    resampled_samples = resample(samples, logits)
    model_samples = resampled_samples.reshape(-1, n_particles, 3)
    # any reference sample
    reference_samples = datamodule.data_test
    chirality_centers = find_chirality_centers(adj_list, atom_types)
    reference_signs = compute_chirality_sign(
        reference_samples.reshape(-1, n_particles, 3)[[1]], chirality_centers
    )
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    print("Symmetry change frac:", (symmetry_change).float().mean())
    model_samples[symmetry_change] *= -1
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    # model_samples = model_samples[~symmetry_change]
    model_samples = model_samples.reshape(-1, n_particles * 3)
    model_sample_energy = datamodule.energy(model_samples)
    # end symmetry check
    if plot_3d:
        import nglview as nv
        import mdtraj as md

        traj_samples_data = md.Trajectory(
            datamodule.unnormalize(model_samples).reshape(-1, n_particles, 3).cpu(),
            topology=datamodule.topology,
        )
        start = model_samples[57513:57514]
        # Langevin
        traj = [start]
        eps = 1e-5
        dt = 1 / 100
        for i in range(100):

            # xt = torch.autograd.graddatamodule.energy(traj[-1])
            with torch.enable_grad():
                x = traj[-1]
                x.requires_grad = True
                target_energy = datamodule.energy(x)
                x_grad = torch.autograd.grad(target_energy.sum(), x)[0].detach()
                # compute the updates
                dX_t = -eps * x_grad * dt + math.sqrt(2 * eps * dt) * torch.randn_like(x)
                print(x.std())
                traj.append(dX_t + x)
        traj = torch.stack(traj)
        return traj_samples_data, model_sample_energy, traj

    x_pred = datamodule.get_phi_psi_vectors(datamodule.unnormalize(model_samples).cpu())
    phis_data, psis_data = x_pred[:, : x_pred.shape[1] // 2], x_pred[:, x_pred.shape[1] // 2 :]
    n_angles = phis_data.shape[1]
    fig, axs = plt.subplots(1, n_angles, figsize=(n_angles * 3, 3), constrained_layout=True)
    plot_range = [-np.pi, np.pi]
    for i in range(phis_data.shape[1]):
        if phis_data.shape[1] == 1:
            ax = axs
        else:
            ax = axs[i]
        h1, x_bins1, y_bins1, im1 = ax.hist2d(
            phis_data[:, i],
            psis_data[:, i],
            100,
            norm=LogNorm(),
            range=[plot_range, plot_range],
            rasterized=True,
        )
        ax.set_xlabel(rf"$\varphi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_ylabel(rf"$\psi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_xticks([-np.pi / 2, np.pi / 2])
        ax.set_xticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
        ax.set_yticks([-np.pi / 2, np.pi / 2])
        ax.set_yticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
    print(f"Saving plot {rama_save_path}")
    plt.savefig(rama_save_path, dpi=300)
    plt.close()
    if False:
        # the tica projection is computed based on reference data
        # the lagtime can be changed in order to get well seperated states
        tica_model = run_tica(
            md.Trajectory(
                datamodule.original_test_data.reshape(-1, n_particles, 3),
                topology=datamodule.topology,
            ),
            lagtime=100,
        )
        # we can then map other data, e.g. generated with the same transformation
        features = tica_features(
            md.Trajectory(
                datamodule.unnormalize(model_samples).cpu().reshape(-1, n_particles, 3),
                topology=datamodule.topology,
            )
        )
        tics = tica_model.transform(features)
        # feat_model = tica_features(traj_samples)
        # tics_model = tica_model.transform(feat_model)
        fig, ax = plt.subplots(figsize=(3, 3), dpi=300, constrained_layout=True)
        ax = plot_tic01(ax, tics, f"MD", tics_lims=tics)
        plt.savefig(tica_save_path, dpi=300)

        plt.close()


# for i, row in df[::-1].iterrows():
##    view = plot_samples(row, overwrite=True, plot_3d=True)
#   if view is not None:
#       break
# view

In [55]:
start = traj_samples_data[57513].xyz

In [31]:
import nglview as nv

In [61]:
df
for i, row in df[
    (df["data/n_particles"] == 63) & df["model/_target_"].str.contains("Normalizing")
].iterrows():
    print(row["data/n_particles"])
    traj_samples_data, energies, traj = plot_samples(row, plot_3d=True, overwrite=True)

    break
view = nv.show_mdtraj(traj_samples_data[:1])
view.clear_representations()
view.add_representation("ball+stick")
view.background = "white"
view

63
63
Symmetry change frac: tensor(0., device='cuda:0')
tensor(1.2932, device='cuda:0', grad_fn=<StdBackward0>)


RuntimeError: you can only change requires_grad flags of leaf variables.

In [35]:
sample_id = 1002


view = nv.show_mdtraj(traj_samples_data[sample_id : sample_id + 100], width="800", height="600")
view.clear_representations()
view.add_representation("ball+stick", selection="protein")
# view.set_size(400,300)
view.background = "white"
view

NGLWidget(max_frame=99)

In [39]:
traj_samples_data_aligned = traj_samples_data.superpose(traj_samples_data, frame=0)
view = nv.show_mdtraj(
    traj_samples_data_aligned[sample_id : sample_id + 100], width="800", height="600"
)
view.clear_representations()
view.add_representation("ball+stick", selection="protein")
# view.set_size(400,300)
view.background = "white"
view

NGLWidget(max_frame=99)

In [40]:
from nglview.contrib.movie import MovieMaker

movie = MovieMaker(view, output="my_gif3.gif", download_folder=".")
# movie.make()
movie.make()

IntProgress(value=0, description='Rendering ...', max=99)

In [11]:
view.add_cartoon(selection="protein")
view.add_licorice("ALA, GLU")
view.clear_representations()

In [12]:
!pip install moviepy

In [13]:
df[df["model/_target_"].str.contains("Normalizing")]
# df["model/_target_"]

,name,Tags,_runtime,_step,_timestamp,_wandb/runtime,epoch,test/1-Wasserstein,test/2-Wasserstein,test/Mean_L1,...,model/net/channels,model/net/img_size,model/net/num_blocks,model/net/patch_size,model/net/in_channels,model/net/layers_per_block,model/data,model/use_com_energy,model/force_gaussian_loss,model/net/pdb_filename
5,silvery-plant-4692,"['jarz', 'jarz_big_v2', 'plot_me']",12916.571724,54,1.738227e+09,12917,0,2.173443,2.213589,0.002599,...,384.0,189.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN
6,light-cosmos-4700,"['jarz', 'jarz_final_iv5', 'plot_me']",6009.650143,8,1.738223e+09,6010,0,0.590164,0.602371,0.001539,...,256.0,66.0,4.0,1.0,1.0,4.0,[{'batch_size': 10000}],1.0,True,NaN
7,different-leaf-4704,"['jarz', 'jarz_final_iv5', 'plot_me']",5750.027659,24,1.738223e+09,5750,0,0.859885,0.873829,0.003170,...,256.0,99.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN
8,wise-durian-4709,"['jarz', 'jarz_final_iv5', 'plot_me']",9923.035325,34,1.738230e+09,9924,0,1.121315,1.146473,0.005909,...,384.0,126.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN


In [14]:
!ffmpeg

ffmpeg version 9c33b2f Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 9.3.0 (crosstool-NG 1.24.0.133_b0863d8_dirty)
  configuration: --prefix=/network/scratch/a/alexander.tong/micromamba/envs/tbg4 --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1627813612080/_build_env/bin/x86_64-conda-linux-gnu-cc --disable-doc --disable-openssl --enable-avresample --enable-gnutls --enable-gpl --enable-hardcoded-tables --enable-libfreetype --enable-libopenh264 --enable-libx264 --enable-pic --enable-pthreads --enable-shared --enable-static --enable-version3 --enable-zlib --enable-libmp3lame --pkg-config=/home/conda/feedstock_root/build_artifacts/ffmpeg_1627813612080/_build_env/bin/pkg-config
  libavutil      56. 51.100 / 56. 51.100
  libavcodec     58. 91.100 / 58. 91.100
  libavformat    58. 45.100 / 58. 45.100
  libavdevice    58. 10.100 / 58. 10.100
  libavfilter     7. 85.100 /  7. 85.100
  libavresample   4.  0.  0 /  4.  0.  0
  libswscale      5.  7.100 /  5.  7.100
  libs

In [15]:
view.add_representation("ball+stick", selection="protein")
view

NGLWidget(max_frame=9)

In [16]:
from src.models.components.utils import (
    check_symmetry_change,
    compute_chirality_sign,
    find_chirality_centers,
    resample,
)

In [17]:
df

,name,Tags,_runtime,_step,_timestamp,_wandb/runtime,epoch,test/1-Wasserstein,test/2-Wasserstein,test/Mean_L1,...,model/net/channels,model/net/img_size,model/net/num_blocks,model/net/patch_size,model/net/in_channels,model/net/layers_per_block,model/data,model/use_com_energy,model/force_gaussian_loss,model/net/pdb_filename
0,proud-lion-3268,"['al', 'cnf', 'eval', 'plot_me', 'v9']",7787.200204,4,1.737968e+09,7787,0,0.658858,0.669871,0.002496,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,prime-waterfall-3273,"['al', 'cnf', 'eval', 'plot_me', 'v9']",22455.632923,17,1.737995e+09,22455,0,0.990239,1.021321,0.001779,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,worthy-cosmos-3414,"['al', 'cnf', 'eval', 'plot_me', 'v9']",15075.665428,25,1.738024e+09,15075,0,1.383969,1.452077,0.004043,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,winter-water-4405,"['dev', 'plot_me']",26766.403887,15854,1.738197e+09,26766,1000,1.141882,1.151315,0.002063,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,effortless-pine-4406,"['dev', 'plot_me']",41786.142418,15950,1.738212e+09,41786,1000,1.613594,1.628095,0.003204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,silvery-plant-4692,"['jarz', 'jarz_big_v2', 'plot_me']",12916.571724,54,1.738227e+09,12917,0,2.173443,2.213589,0.002599,...,384.0,189.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN
6,light-cosmos-4700,"['jarz', 'jarz_final_iv5', 'plot_me']",6009.650143,8,1.738223e+09,6010,0,0.590164,0.602371,0.001539,...,256.0,66.0,4.0,1.0,1.0,4.0,[{'batch_size': 10000}],1.0,True,NaN
7,different-leaf-4704,"['jarz', 'jarz_final_iv5', 'plot_me']",5750.027659,24,1.738223e+09,5750,0,0.859885,0.873829,0.003170,...,256.0,99.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN
8,wise-durian-4709,"['jarz', 'jarz_final_iv5', 'plot_me']",9923.035325,34,1.738230e+09,9924,0,1.121315,1.146473,0.005909,...,384.0,126.0,6.0,1.0,3.0,6.0,[{'batch_size': 10000}],1.0,True,NaN
9,lyric-forest-4732,"['al', 'cnf', 'eval', 'plot_me', 'v9']",4248.781922,41,1.738242e+09,4248,0,2.389132,2.428334,0.008114,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAAAAA.pdb


In [18]:
# npz_file = np.load(f"/result_data/ad2_tbg_full.npz")
def plot_ecnf():
    """
    Plotting for results in format of TBG code.
    """
    from bgflow import MeanFreeNormalDistribution

    npz_file = np.load(f"../old/result_data/Flow-Matching-AD2-amber-weighted-encoding.npz")

    latent_np = npz_file["latent_np"]
    samples_np = npz_file["samples_np"]
    dlogp_np = npz_file["dlogp_np"]
    dim = 66
    n_particles = 22
    prior = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False)
    logp = -prior.energy(torch.from_numpy(latent_np)) - torch.from_numpy(dlogp_np).reshape(-1, 1)
    samples = torch.from_numpy(samples_np)

    cfg = {
        "_target_": "src.data.aldp_datamodule.ALDPDataModule",
        "data_dir": "../data/AD2/",
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": 22,
        "n_dimensions": 3,
        "dim": 66,
    }
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    samples = samples / 10 / datamodule.std
    datamodule.plot_nice_samples(samples, log_p_samples=logp, min_energy=-50, max_energy=25)
    plt.savefig(f"al2_ecnf_density.png", dpi=300)
    plt.close()

    atom_dict = {"C": 0, "H": 1, "N": 2, "O": 3, "S": 4}
    atom_types = []
    for atom_name in datamodule.topology.atoms:
        atom_types.append(atom_name.name[0])
    atom_types = torch.from_numpy(np.array([atom_dict[atom_type] for atom_type in atom_types]))
    n_particles = len(atom_types)
    adj_list = torch.from_numpy(
        np.array(
            [(b.atom1.index, b.atom2.index) for b in datamodule.topology.bonds], dtype=np.int32
        )
    )
    energy_samples = datamodule.energy(samples)
    logits = -energy_samples.flatten() - logp.flatten()
    # energy_samples = datamodule.energy(samples)
    clip = 0.002
    if clip > 0:
        clipped_logits_mask = logits > torch.quantile(logits, 1 - clip)
        logits = logits[~clipped_logits_mask]
        samples = samples[~clipped_logits_mask]
        energy_samples = energy_samples[~clipped_logits_mask]
    # model samples
    resampled_samples = resample(samples, logits)
    model_samples = resampled_samples.reshape(-1, n_particles, 3)
    # any reference sample
    reference_samples = datamodule.data_test
    chirality_centers = find_chirality_centers(adj_list, atom_types)
    reference_signs = compute_chirality_sign(
        reference_samples.reshape(-1, n_particles, 3)[[1]], chirality_centers
    )
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    print("Symmetry change frac:", (symmetry_change).float().mean())
    model_samples[symmetry_change] *= -1
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    # model_samples = model_samples[~symmetry_change]
    model_samples = model_samples.reshape(-1, n_particles * 3)

    x_pred = datamodule.get_phi_psi_vectors(datamodule.unnormalize(model_samples).cpu())
    phis_data, psis_data = x_pred[:, : x_pred.shape[1] // 2], x_pred[:, x_pred.shape[1] // 2 :]
    n_angles = phis_data.shape[1]
    fig, axs = plt.subplots(1, n_angles, figsize=(n_angles * 3, 3), constrained_layout=True)
    plot_range = [-np.pi, np.pi]
    for i in range(phis_data.shape[1]):
        if phis_data.shape[1] == 1:
            ax = axs
        else:
            ax = axs[i]
        h1, x_bins1, y_bins1, im1 = ax.hist2d(
            phis_data[:, i],
            psis_data[:, i],
            100,
            norm=LogNorm(),
            range=[plot_range, plot_range],
            rasterized=True,
        )
        ax.set_xlabel(rf"$\varphi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_ylabel(rf"$\psi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_xticks([-np.pi / 2, np.pi / 2])
        ax.set_xticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
        ax.set_yticks([-np.pi / 2, np.pi / 2])
        ax.set_yticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
    rama_save_path = "al2_ecnf_rama.png"
    print(f"Saving plot {rama_save_path}")
    plt.savefig(rama_save_path, dpi=300)
    plt.close()


plot_ecnf()

Using downloaded and verified file: /tmp/A.pdb
Symmetry change frac: tensor(0.5159)
Saving plot al2_ecnf_rama.png


In [19]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["last-deca"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [20]:
df

,name,Tags,_runtime,_step,_timestamp,_wandb/runtime,epoch,invert/fail_count_0.001,invert/fail_count_0.01,invert/fail_count_sample_0.001,...,callbacks/model_checkpoint_cropped/save_weights_only,callbacks/model_checkpoint_cropped/every_n_train_steps,callbacks/model_checkpoint_cropped/train_time_interval,callbacks/model_checkpoint_cropped/auto_insert_metric_name,callbacks/model_checkpoint_cropped/save_on_train_epoch_end,ckpt_path,task_name,model/params/total,model/params/trainable,model/params/non_trainable
0,lucky-darkness-4902,"['jarz', 'jarz_big', 'last-deca']",16008.912597,91,1.738321e+09,16010,0,361949.90625,1962.500000,2000,...,False,None,None,False,None,/home/ubuntu/scratch/deca/epoch_799_time.ckpt,train,404947040,202473520,202473520
1,distinctive-dew-4903,"['jarz', 'jarz_big', 'last-deca']",15959.445967,91,1.738322e+09,15960,0,362190.03125,1956.199951,2000,...,False,None,None,False,None,/home/ubuntu/scratch/deca/epoch_799_time.ckpt,train,404947040,202473520,202473520


In [21]:
def plot_samples(
    row, clip=0.002, overwrite=False, plot_energy=False, plot_rama=False, plot_3d=False
):
    model = "Unknown"
    if "matching" in row["model/_target_"]:
        if row["model/sigma"] == 0.01:
            model = "ecnf"
        else:
            model = "ecnf++"
    else:
        model = "SBG"
    if model != "SBG":
        print("Skipping non SBG")
        return
    data_dir = row["data/data_dir"]
    n_particles = row["data/n_particles"]
    print(n_particles)
    save_path = f"figs/density_v2/{n_particles}_{model}_density.png"
    rama_save_path = f"figs/rama_v2/{n_particles}_{model}_rama.png"
    tica_save_path = f"figs/tica_v2/{n_particles}_{model}_rama.png"
    if os.path.exists(save_path) and not overwrite:
        print(f"Skipping {save_path}")
        return
    if "bose" in data_dir or "ubuntu" in data_dir:
        data_dir = "../data/alanine"
        if n_particles == 22:
            data_dir = "../data/AD2/"
    try:
        sample_file = torch.load(f"{row['paths/output_dir']}/test_samples.pt", weights_only=True)
        if model == "SBG":
            jarz_sample_file = torch.load(
                f"{row['paths/output_dir']}/test_jarzynski_samples.pt", weights_only=True
            )
    except Exception as e:
        print(e)
        # try loading via joey scratch
        if n_particles == 166:
            residues = 10
        if n_particles == 63:
            residues = 6
        if n_particles == 42:
            residues = 4
        if n_particles == 33:
            residues = 3
        if n_particles < 30:
            residues = 2
        path = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_samples.pt"
        jpath = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_jarzynski_samples.pt"
        sample_file = torch.load(path, weights_only=True)
        jarz_sample_file = torch.load(jpath, weights_only=True)
        # print(e)
        # print(f"File not found {row['paths/output_dir']}/test_samples.pt. Skipping.")

    samples = sample_file["samples"]
    logp = sample_file["log_p"]
    jarzynski_samples = None
    if model == "SBG":
        jarzynski_samples = jarz_sample_file["samples"]
        jarzynski_logits = jarz_sample_file["logits"]
        clip = 0
    cfg = {
        "_target_": row["data/_target_"],
        "data_dir": data_dir,
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": int(row["data/n_particles"]),
        "n_dimensions": row["data/n_dimensions"],
        "dim": int(row["data/dim"]),
    }

    if n_particles > 22 and row["data/filename"] == row["data/filename"]:
        cfg["filename"] = row["data/filename"]
        # cfg["pdb_filename"] = row["data/pdb_filename"]
    print(cfg)
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    min_energy = -50
    max_energy = 20
    ylim = (0, 0.1)
    if n_particles == 166:
        min_energy = -500
        max_energy = 200
        ylim = (0, 0.04)
    if n_particles == 63:
        min_energy = -130
        max_energy = -40
        ylim = (0, 0.1)
    if n_particles == 53:
        min_energy = -150
        max_energy = -60
        ylim = (0, 0.1)
    if n_particles == 42:
        min_energy = -20
        max_energy = 80
        ylim = (0, 0.1)
    if n_particles == 33:
        min_energy = -200
        max_energy = -100
        ylim = (0, 0.1)
    # max_energy = 100
    if plot_energy:
        datamodule.plot_nice_samples(
            samples,
            log_p_samples=logp,
            samples_jarzynski=jarzynski_samples,
            min_energy=min_energy,
            max_energy=max_energy,
            # clip_energy=max_energy,
            ylim=ylim,
        )
        print(f"saving plot to {save_path}")
        plt.savefig(save_path, dpi=300)
        plt.close()

    # ramaplot
    from matplotlib.colors import LogNorm

    # symmetry check
    from src.models.components.utils import (
        check_symmetry_change,
        compute_chirality_sign,
        find_chirality_centers,
        resample,
    )

    atom_dict = {"C": 0, "H": 1, "N": 2, "O": 3, "S": 4}
    atom_types = []
    for atom_name in datamodule.topology.atoms:
        atom_types.append(atom_name.name[0])
    atom_types = torch.from_numpy(np.array([atom_dict[atom_type] for atom_type in atom_types]))
    n_particles = len(atom_types)
    adj_list = torch.from_numpy(
        np.array(
            [(b.atom1.index, b.atom2.index) for b in datamodule.topology.bonds], dtype=np.int32
        )
    )
    if model == "SBG":
        samples = jarzynski_samples
        logits = jarzynski_logits
    else:
        logits = -energy_samples.flatten() - logp.flatten()
    print(datamodule.data_test.shape)
    energy_samples = datamodule.energy(samples)
    if clip > 0:
        clipped_logits_mask = logits > torch.quantile(logits, 1 - clip)
        logits = logits[~clipped_logits_mask]
        samples = samples[~clipped_logits_mask]
        energy_samples = energy_samples[~clipped_logits_mask]
    # model samples
    resampled_samples = resample(samples, logits)
    model_samples = resampled_samples.reshape(-1, n_particles, 3)
    # any reference sample
    reference_samples = datamodule.data_test
    chirality_centers = find_chirality_centers(adj_list, atom_types)
    reference_signs = compute_chirality_sign(
        reference_samples.reshape(-1, n_particles, 3)[[1]], chirality_centers
    )
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    print("Symmetry change frac:", (symmetry_change).float().mean())
    model_samples[symmetry_change] *= -1
    symmetry_change = check_symmetry_change(model_samples, chirality_centers, reference_signs)
    # model_samples = model_samples[~symmetry_change]
    model_samples = model_samples.reshape(-1, n_particles * 3)
    model_sample_energy = datamodule.energy(model_samples)
    # end symmetry check
    if plot_3d:
        import nglview as nv
        import mdtraj as md

        traj_samples_data = md.Trajectory(
            datamodule.unnormalize(model_samples).reshape(-1, n_particles, 3).cpu(),
            topology=datamodule.topology,
        )
        return traj_samples_data, model_sample_energy

    x_pred = datamodule.get_phi_psi_vectors(datamodule.unnormalize(model_samples).cpu())
    phis_data, psis_data = x_pred[:, : x_pred.shape[1] // 2], x_pred[:, x_pred.shape[1] // 2 :]
    n_angles = phis_data.shape[1]
    fig, axs = plt.subplots(1, n_angles, figsize=(n_angles * 3, 3), constrained_layout=True)
    plot_range = [-np.pi, np.pi]
    for i in range(phis_data.shape[1]):
        if phis_data.shape[1] == 1:
            ax = axs
        else:
            ax = axs[i]
        h1, x_bins1, y_bins1, im1 = ax.hist2d(
            phis_data[:, i],
            psis_data[:, i],
            100,
            norm=LogNorm(),
            range=[plot_range, plot_range],
            rasterized=True,
        )
        ax.set_xlabel(rf"$\varphi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_ylabel(rf"$\psi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_xticks([-np.pi / 2, np.pi / 2])
        ax.set_xticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
        ax.set_yticks([-np.pi / 2, np.pi / 2])
        ax.set_yticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
    print(f"Saving plot {rama_save_path}")
    plt.savefig(rama_save_path, dpi=300)
    plt.close()
    if False:
        # the tica projection is computed based on reference data
        # the lagtime can be changed in order to get well seperated states
        tica_model = run_tica(
            md.Trajectory(
                datamodule.original_test_data.reshape(-1, n_particles, 3),
                topology=datamodule.topology,
            ),
            lagtime=100,
        )
        # we can then map other data, e.g. generated with the same transformation
        features = tica_features(
            md.Trajectory(
                datamodule.unnormalize(model_samples).cpu().reshape(-1, n_particles, 3),
                topology=datamodule.topology,
            )
        )
        tics = tica_model.transform(features)
        # feat_model = tica_features(traj_samples)
        # tics_model = tica_model.transform(feat_model)
        fig, ax = plt.subplots(figsize=(3, 3), dpi=300, constrained_layout=True)
        ax = plot_tic01(ax, tics, f"MD", tics_lims=tics)
        plt.savefig(tica_save_path, dpi=300)

        plt.close()


# for i, row in df[::-1].iterrows():
##    view = plot_samples(row, overwrite=True, plot_3d=True)
#   if view is not None:
#       break
# view
for i, row in df.iterrows():
    plot_samples(row, overwrite=True, plot_energy=True, plot_3d=True)

166
[Errno 2] No such file or directory: '/home/ubuntu/scratch/logs/train/runs/2025-01-31_06-31-34/test_samples.pt'


FileNotFoundError: [Errno 2] No such file or directory: '/network/scratch/b/bosejoey/fast-tbg/al10/test_samples.pt'

In [ ]:
energies[torch.argmin(energies)]
# torch.argmin(energies)

In [ ]:
for i, row in df.iterrows():
    print(row["data/n_particles"])
    traj_samples_data, energies = plot_samples(row, plot_3d=True, overwrite=True)

    break
view = nv.show_mdtraj(traj_samples_data[7247:7248])
view.clear_representations()
# view.add_representation("ball+stick")
view.add_representation("cartoon")
view

In [ ]:
view.clear_representations()
#view.add_representation("ball+stick")
view.add_representation("cartoon")
view

In [ ]:
view.clear_representations()
view.add_representation("ball+stick")
# view.add_representation("cartoon")
view

In [ ]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["plot_me"]},
                #'group': {'$in': ['5_vars']},
                "config.data.n_particles": {"$eq": 33},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [ ]:
df.iloc[:1]

In [ ]:
def resample(samples, logits, size=None):
    """
    Resample samples with given logits.
    Args:
        samples: Samples to resample
        logits: Logits for resampling
    Returns:
        Resampled samples
    """
    probs = torch.softmax(logits, dim=-1)
    if size is None:
        size = samples.size(0)
    resampled_samples = torch.multinomial(probs, size, replacement=True)
    return samples[resampled_samples]


def plot_samples(
    row, clip=0.002, overwrite=False, plot_energy=False, plot_rama=False, plot_3d=False
):
    model = "Unknown"
    if "matching" in row["model/_target_"]:
        if row["model/sigma"] == 0.01:
            model = "ecnf"
        else:
            model = "ecnf++"
    else:
        model = "SBG"
    # if model != "SBG":
    #    print("Skipping non SBG")
    #    return
    data_dir = row["data/data_dir"]
    n_particles = row["data/n_particles"]
    save_path = f"figs/density_v2/{n_particles}_{model}_density.png"
    rama_save_path = f"figs/rama_v2/{n_particles}_{model}_rama.png"
    tica_save_path = f"figs/tica_v2/{n_particles}_{model}_rama.png"
    if os.path.exists(save_path) and not overwrite:
        print(f"Skipping {save_path}")
        return
    if "bose" in data_dir or "ubuntu" in data_dir:
        data_dir = "../data/alanine"
        if n_particles == 22:
            data_dir = "../data/AD2/"
    try:
        sample_file = torch.load(f"{row['paths/output_dir']}/test_samples.pt", weights_only=True)
        if model == "SBG":
            jarz_sample_file = torch.load(
                f"{row['paths/output_dir']}/test_jarzynski_samples.pt", weights_only=True
            )
    except Exception as e:
        # try loading via joey scratch
        if n_particles == 166:
            residues = 10
        if n_particles == 63:
            residues = 6
        if n_particles == 42:
            residues = 4
        if n_particles == 33:
            residues = 3
        if n_particles < 30:
            residues = 2
        path = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_samples.pt"
        jpath = f"/network/scratch/b/bosejoey/fast-tbg/al{residues}/test_jarzynski_samples.pt"
        sample_file = torch.load(path, weights_only=True)
        jarz_sample_file = torch.load(jpath, weights_only=True)
        # print(e)
        # print(f"File not found {row['paths/output_dir']}/test_samples.pt. Skipping.")

    samples = sample_file["samples"]
    logp = sample_file["log_p"]
    jarzynski_samples = None
    if model == "SBG":
        jarzynski_samples = jarz_sample_file["samples"]
        jarzynski_logits = jarz_sample_file["logits"]
        clip = 0
    cfg = {
        "_target_": row["data/_target_"],
        "data_dir": data_dir,
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": int(row["data/n_particles"]),
        "n_dimensions": row["data/n_dimensions"],
        "dim": int(row["data/dim"]),
    }

    if (
        n_particles > 22
        and "data/filename" in row
        and row["data/filename"] == row["data/filename"]
    ):
        cfg["filename"] = row["data/filename"]
        # cfg["pdb_filename"] = row["data/pdb_filename"]
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    min_energy = -50
    max_energy = 20
    ylim = (0, 0.1)
    if n_particles == 166:
        min_energy = -500
        max_energy = 200
        ylim = (0, 0.04)
    if n_particles == 63:
        min_energy = -130
        max_energy = -40
        ylim = (0, 0.1)
    if n_particles == 53:
        min_energy = -150
        max_energy = -60
        ylim = (0, 0.1)
    if n_particles == 42:
        min_energy = -20
        max_energy = 80
        ylim = (0, 0.1)
    if n_particles == 33:
        min_energy = -200
        max_energy = -100
        ylim = (0, 0.1)
    # max_energy = 100
    if plot_energy:
        datamodule.plot_nice_samples(
            samples,
            log_p_samples=logp,
            samples_jarzynski=jarzynski_samples,
            min_energy=min_energy,
            max_energy=max_energy,
            # clip_energy=max_energy,
            ylim=ylim,
        )
        print(f"saving plot to {save_path}")
        plt.savefig(save_path, dpi=300)
        plt.close()

    # ramaplot
    from matplotlib.colors import LogNorm

    # symmetry check
    from src.models.components.utils import (
        check_symmetry_change,
        compute_chirality_sign,
        find_chirality_centers,
        resample,
    )

    atom_dict = {"C": 0, "H": 1, "N": 2, "O": 3, "S": 4}
    atom_types = []
    for atom_name in datamodule.topology.atoms:
        atom_types.append(atom_name.name[0])
    atom_types = torch.from_numpy(np.array([atom_dict[atom_type] for atom_type in atom_types]))
    n_particles = len(atom_types)
    adj_list = torch.from_numpy(
        np.array(
            [(b.atom1.index, b.atom2.index) for b in datamodule.topology.bonds], dtype=np.int32
        )
    )
    energy_samples = datamodule.energy(samples)
    if model == "SBG":
        samples = jarzynski_samples
        logits = jarzynski_logits
    else:
        logits = -energy_samples.flatten() - logp.flatten()

    if clip > 0:
        clipped_logits_mask = logits > torch.quantile(logits, 1 - clip)
        logits = logits[~clipped_logits_mask]
        samples = samples[~clipped_logits_mask]
        energy_samples = energy_samples[~clipped_logits_mask]
    # model samples
    results = []
    for seed in range(3):
        for n in [1000, 2000, 5000, 10000]:
            resampled_samples = resample(samples[:n], logits[:n])
            # model_samples = resampled_samples.reshape(-1, n_particles, 3)
            if False:
                # any reference sample
                reference_samples = datamodule.data_test
                chirality_centers = find_chirality_centers(adj_list, atom_types)
                reference_signs = compute_chirality_sign(
                    reference_samples.reshape(-1, n_particles, 3)[[1]], chirality_centers
                )
                symmetry_change = check_symmetry_change(
                    model_samples, chirality_centers, reference_signs
                )
                # print("Symmetry change frac:", (symmetry_change).float().mean())
                model_samples[symmetry_change] *= -1
                symmetry_change = check_symmetry_change(
                    model_samples, chirality_centers, reference_signs
                )
                # model_samples = model_samples[~symmetry_change]
                model_samples = model_samples.reshape(-1, n_particles * 3)
            # model_sample_energy = datamodule.energy(model_samples)
            # from src.data.components.distribution_distances import energy_distances

            # dists = energy_distances(
            #    model_sample_energy.flatten(), datamodule.energy(datamodule.data_test).flatten()
            # )

            dists = datamodule.get_ramachandran_metrics(
                datamodule.unnormalize(resampled_samples).cpu()
            )
            print(n, seed, dists["/torus_wasserstein"])
            results.append((n, seed, dists["/torus_wasserstein"]))
    return results


for i, row in df.iloc[:1].iterrows():

    results = plot_samples(row, overwrite=True, plot_energy=True, plot_3d=True)

In [ ]:
results

In [ ]:
arr = np.array([0.01741, 0.01143, 0.017473])
print(arr.mean(), arr.std())

In [ ]:
arr = np.array([9.41204, 8.04074, 9.40845])
print(arr.mean(), arr.std())
arr2 = np.array([5.35238, 5.50246, 5.35948])

print(arr2.mean(), arr2.std())